# 05. Multi-head attention: отдельные головы

Лекция: [Multi-head attention](../../notes/interview-prep/08d-multi-head-attention.md).

Сегодня выполни только **остановку A**: запусти две головы и объедини выходы. Затем разберём выходную проекцию и nn.Module (B), проверки (C). Реализация с отдельными объектами намеренно проще векторизованной.

Сохраняем causal mask. Без bias, dropout, padding, residual и LayerNorm. Вход — готовые векторы X, не token IDs.

## A1. Что изменилось в одной голове

Раньше проекции были D→D. Теперь у каждой головы D→d_head. При D=4 и H=2 выберем d_head=2.

**Каждая голова получает весь X формы (B,T,4)**. Мы не отдаём одной голове первые два признака, а другой последние два. Каждая сама учится преобразовывать все четыре входные координаты в две.

Ниже готовая адаптация твоего causal attention. Изменён размер выхода Linear; делим по-прежнему на корень из размерности Query.

In [1]:
import math

import torch
from torch import nn

torch.manual_seed(42)
torch.set_printoptions(precision=3, sci_mode=False)


class CausalAttentionHead(nn.Module):
    def __init__(self, d_model, head_dim):
        super().__init__()
        self.query_projection = nn.Linear(d_model, head_dim, bias=False)
        self.key_projection = nn.Linear(d_model, head_dim, bias=False)
        self.value_projection = nn.Linear(d_model, head_dim, bias=False)

    def forward(self, x):
        q = self.query_projection(x)
        k = self.key_projection(x)
        v = self.value_projection(x)
        scores = (q @ k.transpose(-2, -1)) / math.sqrt(q.shape[-1])
        positions = torch.arange(x.shape[1], device=x.device)
        blocked = positions[None, :] > positions[:, None]
        weights = torch.softmax(scores.masked_fill(blocked, float('-inf')), dim=-1)
        return weights @ v


B, T, D, H = 2, 3, 4, 2
head_dim = D // H
x = torch.randn(B, T, D)
head_0 = CausalAttentionHead(D, head_dim)
head_1 = CausalAttentionHead(D, head_dim)

print('X:', x.shape)
print('Query weight одной головы:', head_0.query_projection.weight.shape)
print('Параметров одной головы:', sum(p.numel() for p in head_0.parameters()))

X: torch.Size([2, 3, 4])
Query weight одной головы: torch.Size([2, 4])
Параметров одной головы: 24


## A2. Одинаковый вход — разные проекции

Вызови обе головы на одном x. Они созданы двумя вызовами конструктора и имеют независимые параметры. Разные результаты возможны, но не обязаны различаться для любых весов и входов. Им не заданы роли вроде «грамматика» и «смысл».

In [2]:
output_0 = head_0(x)
output_1 = head_1(x)

assert isinstance(output_0, torch.Tensor)
assert isinstance(output_1, torch.Tensor)
assert output_0.shape == output_1.shape == (B, T, head_dim)
assert head_0.query_projection.weight is not head_1.query_projection.weight
print('Голова 0, первая последовательность:')
print(output_0[0])
print('Голова 1, первая последовательность:')
print(output_1[0])

Голова 0, первая последовательность:
tensor([[1.392, 0.055],
        [1.109, 0.312],
        [0.602, 0.299]], grad_fn=<SelectBackward0>)
Голова 1, первая последовательность:
tensor([[-0.340,  1.484],
        [ 0.197,  1.173],
        [ 0.049,  0.807]], grad_fn=<SelectBackward0>)


## A3. Склеиваем, а не складываем

Для одной позиции пусть головы вернули [1,2] и [10,20].

```text
Сложение:     [11,22]        — 2 координаты
Конкатенация: [1,2,10,20]    — 4 координаты
```

`torch.cat([a,b], dim=...)` соединяет вдоль существующей оси. Нам нужно объединить признаки одной и той же позиции, сохранив B и T. Подумай, какую ось указать.

В отличие от cat, stack добавил бы новую ось. Пока отдельную ось голов не вводим.

In [3]:
combined = torch.cat([output_0, output_1], dim=-1)

assert isinstance(combined, torch.Tensor)
assert combined.shape == (B, T, D)
torch.testing.assert_close(combined[..., :head_dim], output_0)
torch.testing.assert_close(combined[..., head_dim:], output_1)
print('Combined:', combined.shape)
print('Первая позиция:', combined[0, 0])

Combined: torch.Size([2, 3, 4])
Первая позиция: tensor([ 1.392,  0.055, -0.340,  1.484], grad_fn=<SelectBackward0>)


### Остановка A — сохрани и пришли на проверку

1. При D=4, H=2 какую форму имеет Query-матрица одной головы?
2. Каждая голова видит весь вектор X или только половину? Почему?
3. Каковы формы output_0, output_1 и combined при B=2, T=3?
4. Почему combined получается через cat, а не сложение?
5. Должны ли две головы заранее отвечать за конкретные языковые связи?

**Мои ответы:**

1. 2,4
2. Весь вектор. Потому что применяется ко всей размерности эмбединга и к каждому токену
3. output_0 = output_1 = (2, 3, 2), combined=(2,3,4)
4. Потому что он должен собрать признаки голов, а не перемешать их. Но почему именно - я не понял сам внутренний смысл
5. Нет, в процессе обучения это как-то само определяется, но я пока не понимаю как

**Разбор ментора (исходные ответы сохранены):**

1. Верно: Query-матрица одной головы имеет форму (head_dim,d_model) = (2,4).
2. Верно: каждая голова получает весь X. Её Linear(4,2) может использовать все четыре входные координаты для каждой из двух выходных. Параметры одной головы общие для позиций, но отличаются от параметров другой головы.
3. Формы верны. Уточнение записи: равны формы output_0.shape и output_1.shape, а не обязательно значения output_0 и output_1. После cat форма combined — (2,3,4).
4. Интуиция верная, но причина в сохранении отдельных групп признаков до обучаемого смешивания. Пары [1,2]+[3,4] и [0,0]+[4,6] дают одну сумму [4,6], поэтому по сумме эти варианты не различить. Конкатенация даёт [1,2,3,4] и [0,0,4,6], сохраняя различие. Следующая выходная проекция учится использовать признаки разных голов с разными коэффициентами. Сложение математически возможно, но это другая конструкция.
5. Верно: заранее роли не назначены. Разные начальные параметры дают разные вычисления; общий loss через выходную проекцию и cat даёт градиенты каждой голове, а оптимизатор обновляет её параметры. Специализация не гарантирована: головы могут частично дублироваться. Пока модель не обучалась языку, различия случайных выходов не доказывают языковых связей.

**Дополнение: что означает dim=-1.** Это последняя ось, а не последний элемент. Для выхода головы (B,T,head_dim) оси 0,1,2 означают batch, позиции, признаки; dim=-1 здесь равен dim=2. cat по этой оси объединяет признаки одной позиции, сохраняя B и T. Для двух тензоров (2,3,2): cat по dim=0 даёт (4,3,2), по dim=1 — (2,6,2), по dim=-1 — (2,3,4). В scores формы (B,T,T) последняя ось означает источники Keys, поэтому смысл dim зависит от тензора.

Самопроверка в чате: два выхода (8,20,16), cat по dim=-1. Ответ ученика (8,20,32) — верный.

Часть A проверена по сохранённым результатам. Далее часть B: обучаемое смешивание результатов голов.

## B. Собираем модуль и добавляем выходную проекцию

После cat вектор состоит из блоков признаков разных голов. Выходная проекция nn.Linear(D,D,bias=False) обучаемо смешивает эти признаки для каждой позиции. Одинаковая форма входа/выхода не делает этот слой тождественным.

Несколько голов храним в nn.ModuleList, чтобы параметры всех голов регистрировались в модели. Обычный Python list сам по себе этого не делает. ModuleList не имеет готового forward: перебираем его и вызываем каждую голову.

Важно: создавай новую голову на каждой итерации, а не повторяй один объект через [head] * H.

In [4]:
class MultiHeadCausalAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        if d_model <= 0 or num_heads <= 0 or d_model % num_heads != 0:
            raise ValueError('Нужны положительные D,H и D, кратное H')
        self.d_model = d_model
        head_dim = d_model // num_heads

        self.heads = nn.ModuleList([
            CausalAttentionHead(d_model, head_dim)
            for _ in range(num_heads)
        ])
        self.output_projection = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x):
        if x.ndim != 3 or x.shape[-1] != self.d_model or x.shape[1] == 0:
            raise ValueError('Ожидается непустой x формы (B,T,d_model)')
        head_outputs = [head(x) for head in self.heads]
        combined = torch.cat(head_outputs, dim=-1)
        return self.output_projection(combined)

In [5]:
model = MultiHeadCausalAttention(d_model=D, num_heads=H)
assert isinstance(model.heads, nn.ModuleList)
assert len(model.heads) == H
assert isinstance(model.output_projection, nn.Linear)
assert model.output_projection.in_features == model.output_projection.out_features == D
assert model.output_projection.bias is None

# Переносим уже созданные головы для проверки на ОДНИХ И ТЕХ ЖЕ параметрах.
model.heads[0].load_state_dict(head_0.state_dict())
model.heads[1].load_state_dict(head_1.state_dict())
result = model(x)
expected = model.output_projection(torch.cat([head_0(x), head_1(x)], dim=-1))
torch.testing.assert_close(result, expected)
assert result.shape == (B, T, D)
assert sum(p.numel() for p in model.parameters()) == 4 * D * D == 64
assert len(list(model.parameters())) == 3 * H + 1
print('Форма, регистрация параметров и эталон: OK')
print('Число параметров:', sum(p.numel() for p in model.parameters()))

Форма, регистрация параметров и эталон: OK
Число параметров: 64


### Остановка B

1. Зачем ModuleList вместо обычного списка?
2. Зачем нужна выходная проекция, если после cat уже получилась размерность D?
3. Почему здесь 64 параметра? Посчитай отдельно головы и выходную проекцию.
4. Изменится ли число параметров при H=4 и том же D=4 в этой архитектуре?

**Мои ответы:**

1. Чтобы putorch корректно учитывал параметры каждого модуля в списке
2. Чтобы "перемешать" информацию из голов. Так мы сможем извлекать информацию из всех голов в обучаемом полносвязном слое
3. Головы 2\*4\*2\*3=48, проекция 4\*4=16, получилось 64
4. Будет столько же, 4*1*3*4=48, и 16 - проекция

**Разбор ментора (исходные ответы сохранены):**

Ошибка кода: в выражении [self.heads[h](x) for h in self.heads] переменная h уже является объектом CausalAttentionHead, а не целочисленным индексом. Поэтому self.heads[h] вызывает TypeError. Нужно вызвать сам объект: [head(x) for head in self.heads]. Эквивалентный явный цикл: создать пустой список, перебрать for head in self.heads и добавлять head(x). Код оставлен ученику для исправления и повторного запуска.

1. Верно. ModuleList регистрирует дочерние головы, поэтому parameters(), state_dict() и перенос устройства учитывают их параметры.
2. Верно. Выходная проекция обучаемо смешивает признаки всех голов для каждой позиции. Сохранение размерности D не делает её тождественной операцией.
3. Пропущен множитель 3: в каждой голове три матрицы — Q, K и V. Одна матрица: 2*4=8 параметров; одна голова: 3*8=24; две головы: 2*24=48. Выходная проекция: 4*4=16. Итого 64, все слои без bias.
4. При H=4, D=4 размер головы станет 1. Одна голова: 3*4*1=12; четыре головы: 48; выходная проекция: 16. Итого снова 64. В этой конструкции без bias при head_dim=D/H общее число параметров равно 4*D*D. Число матриц внимания при этом растёт с H; неизменное число параметров не означает неизменные затраты памяти.

После исправления перезапусти определение класса и создай новый model, затем выполни проверки. Сохранённый traceback содержит старый return combined, тогда как текущий исходник уже вызывает output_projection: исходник и выполненное состояние ядра расходились. Restart Kernel → Run All до конца B позволит проверить именно текущий код.

## C. Проверки: размеры, причинность, градиенты

Готовые тесты. Обучение на искусственном loss не является обучением языку. Проверки градиентов выполняются на последовательности длины больше 1: при T=1 attention всегда равен 1, поэтому градиенты Q/K могут быть нулевыми.

In [6]:
for test_B, test_T in [(1, 1), (3, 5)]:
    test_x = torch.randn(test_B, test_T, D)
    test_y = model(test_x)
    assert test_y.shape == test_x.shape
    assert torch.isfinite(test_y).all()

with torch.no_grad():
    changed = x.clone()
    changed[:, -1] += torch.tensor([10., -7., 5., 3.], device=x.device)
    torch.testing.assert_close(model(x)[:, :-1], model(changed)[:, :-1])
    # Выходная проекция смешивает признаки, но не позиции.
    first_values = torch.cat([h.value_projection(x[:, :1]) for h in model.heads], dim=-1)
    torch.testing.assert_close(model(x)[:, :1], model.output_projection(first_values))
print('Разные B/T и причинность: OK')

model.zero_grad(set_to_none=True)
probe_loss = model(x).square().mean()
probe_loss.backward()
for name, p in model.named_parameters():
    assert p.grad is not None
    assert torch.isfinite(p.grad).all()
    print(name, 'grad norm:', p.grad.norm().item())
print('Градиенты всех зарегистрированных параметров: OK')

Разные B/T и причинность: OK
heads.0.query_projection.weight grad norm: 0.0056066568940877914
heads.0.key_projection.weight grad norm: 0.003817421616986394
heads.0.value_projection.weight grad norm: 0.39007166028022766
heads.1.query_projection.weight grad norm: 0.006027194671332836
heads.1.key_projection.weight grad norm: 0.006243527866899967
heads.1.value_projection.weight grad norm: 0.23514746129512787
output_projection.weight grad norm: 0.4605954587459564
Градиенты всех зарегистрированных параметров: OK


### Остановка C

1. Почему output_projection не нарушает causal mask?
2. Какие параметры получили градиенты? Изменились ли веса без optimizer.step()?
3. Что именно доказали тесты, а что они не говорят о качестве будущей языковой модели?

**Мои ответы:**

1. Потому что в него уже не попадает информация о будущих токенах, так как мы ее отсеили для каждой головы
2. Не знаю, веса не изменились
3. Не знаю

**Разбор ментора (исходные ответы сохранены):**

1. Верно, с важным уточнением: выходная проекция работает отдельно с вектором каждой позиции по последней оси и не смешивает позиции. Головы уже исключили будущее, а позиционный Linear не добавляет новых связей между токенами. Само по себе отсутствие будущего в выходах голов не оправдывало бы любое последующее преобразование: смешивание разных позиций могло бы снова создать утечку.
2. Градиенты получили семь матриц: Q/K/V головы 0, Q/K/V головы 1 и выходная проекция. Это видно по named_parameters() и сохранённым outputs. backward вычислил и накопил градиенты в p.grad, но веса без optimizer.step() не изменились — эта часть ответа верная.
3. На конкретных тестовых входах проверены форма и конечность выхода, отсутствие влияния изменённого будущего на прошлое, правильный результат первой позиции, наличие конечных градиентов у всех зарегистрированных параметров. Нормы в этом запуске ненулевые, но проверка не требует ненулевых градиентов всегда. Это не доказательство качества языка, обучения на корпусе или корректности на абсолютно всех возможных входах.

**Как читать проверочную ячейку:**

- Цикл по [(1,1),(3,5)] проверяет одну последовательность из одного токена и три последовательности по пять токенов. randn создаёт случайные входные векторы, assert проверяет форму и отсутствие NaN/inf.
- x.clone() создаёт копию; changed[:, -1] выбирает последний токен каждой последовательности. Прибавление вектора меняет только эти токены, оставляя исходный x прежним.
- [:, :-1] выбирает все позиции, кроме последней. assert_close сравнивает прошлые выходы до и после изменения будущего с численным допуском. no_grad здесь отключает построение графа, а не обеспечивает причинность.
- x[:, :1] выбирает первую позицию с сохранением оси T: (B,1,D), в отличие от x[:,0] формы (B,D). Каждая голова в первой позиции видит только собственный Value с весом 1. Поэтому вручную получаем эти Values, склеиваем и применяем выходную проекцию; сравниваем с первым выходом полной модели.
- model.zero_grad(set_to_none=True) очищает предыдущие градиенты. model(x).square().mean() — искусственный скалярный loss, среднее квадратов всех элементов выхода; backward проверяет связность графа. Это не loss предсказания следующего токена.
- named_parameters() выдаёт имя и объект каждой матрицы параметров. p.grad is not None проверяет наличие градиента; isfinite(...).all() — отсутствие NaN/inf. grad.norm().item() выводит одно число: евклидову норму всех элементов градиента. Для матрицы это норма Фробениуса, а не метрика качества модели.

После всех этапов: Restart Kernel → Run All → сохранить outputs. Этот вариант использует цикл по головам. Следом сопоставим его с векторизованным вариантом (B,H,T,d_head); ускорение заранее не заявляем без замеров.